# 03 — Vanishing and exploding gradients

This is the pivot of the chapter. In notebook 2 we stacked two or three layers and they trained
without trouble — but the whole bet of *deep* learning is **many** layers. Stack ten the naive way
and something breaks: the network stops learning, its loss flat from the very first epoch.

This notebook is about *why*. We will measure the problem with our own hands — watch the gradient
either **collapse to nothing** or **blow up** as it travels back through the layers — and then see it
stall a real network. Understanding this one mechanism is the key that unlocks everything after it:
initialization, normalization, the modern recipe all exist because of what we find here.

**Prerequisites**
- Notebooks 1–2 — the `L`-layer network we built by hand (forward pass, backward pass, training loop).
- Chapter 11, NB 3 — backpropagation as the chain rule, layer by layer.
- Chapter 11, NB 4 — sigmoid and tanh **saturate**: their flat tails produce tiny gradients.

**What you'll be able to do**
- Explain why a gradient shrinks or grows as it is back-propagated through many layers.
- Measure the per-layer gradient by hand and *see* it vanish or explode.
- Recognize the failure in a real training run — a loss curve that never moves.

## The pivot

Notebook 2 left us with a puzzle. Depth *should* be powerful — each layer remaps the data for the
next — yet when we hinted at stacking *many* layers, we warned that naive deep nets get fragile. Here
is the concrete failure: take a network ten layers deep, initialize it the way we have all along, and
train it. The loss does not move. The network is stuck at its random starting point, epoch after
epoch.

The cause is not the optimizer and not the data. It is something that happens **inside the backward
pass** when you chain many layers together — and once you see it, the rest of the chapter (how to
fix it) falls into place.

## The mechanism: the chain rule across many layers

Recall how backpropagation works (chapter 11, NB 3): the gradient is carried **backward**, one layer
at a time, and at each layer it is multiplied by a factor — roughly the layer's weights times its
activation's slope (`W · activation'`).

Now follow the gradient all the way down to an **early** layer. To get there it had to pass through
*every* layer above it, picking up one multiplicative factor at each. So the gradient at an early
layer is a **product of many factors**:

`gradient(early layer)  ≈  factor_L · factor_(L−1) · ... · factor_2 · factor_1`

Multiply many numbers together and there are only two generic outcomes:

- if each factor is **less than 1**, the product shrinks toward **0** — the gradient **vanishes**;
- if each factor is **greater than 1**, the product runs off to **infinity** — the gradient **explodes**.

The deeper the network, the more factors in the product, and the more violent the outcome. Let's draw
it, then measure it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_circles
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

from ml_course import colors, viz

viz.use_course_style()

SEED = 0

# A schematic: the backward signal starts at 1.0 (right) and is multiplied by the same factor at
# each layer as it travels left. Two regimes: factor 0.3 (vanish) and factor 2.0 (explode).
steps = np.arange(7)  # 6 layers
vanish = 0.3 ** steps
explode = 2.0 ** steps

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(steps, vanish, "o-", color=colors.COLORS["model"], lw=2,
        label="factor 0.3 each layer (vanish)")
ax.plot(steps, explode, "s-", color=colors.COLORS["error"], lw=2,
        label="factor 2.0 each layer (explode)")
ax.axhline(1.0, color=colors.COLORS["muted"], ls="--", lw=1)
for s in steps:
    ax.annotate(f"{vanish[s]:.2g}", (s, vanish[s]), textcoords="offset points",
                xytext=(0, -14), ha="center", fontsize=8, color=colors.COLORS["model"])
    ax.annotate(f"{explode[s]:.0f}", (s, explode[s]), textcoords="offset points",
                xytext=(0, 8), ha="center", fontsize=8, color=colors.COLORS["error"])
ax.set_yscale("log")
ax.set_xlabel("layers the gradient has passed through")
ax.set_ylabel("size of the backward signal (log scale)")
ax.set_title("A chain of factors: the gradient vanishes or explodes with depth")
ax.legend()
plt.show()

**Read the figure.** The backward signal starts at 1 (right edge) and is multiplied by the same
factor at every layer. With a factor below 1 (here 0.3, blue) the signal is more than halved at
each step and after six layers is already near `1e-3` — and it keeps falling. With a factor above 1
(here 2.0, coral) it doubles each step and runs away. The same arithmetic that lets depth *compose*
representations (notebook 2) also **compounds the gradient** — a chain of small factors drives it to
zero, a chain of large ones lets it run away. Real networks are not fixed at 0.3 or 2.0, so let's
measure what actually happens.

## Measuring the real thing

We reuse the network we already built — the same `L`-layer forward and backward pass from notebooks 1
and 2 — with two small additions: a **sigmoid** hidden-activation option (alongside ReLU), and a knob
for the **initial weight scale**. Then we build a deep stack of **ten hidden layers**, run a single
forward and backward pass, and read the size of the gradient at *each* layer.

Two setups tell the whole story:

- **sigmoid activations with small weights** (the way chapter 11 and notebook 1 initialized);
- **ReLU activations with large weights** (standard-normal, scale 1).

In [ ]:
def init_params(sizes, scale, seed=SEED):
    # sizes = [n_in, h1, ..., n_out]; every weight ~ Normal(0, scale^2). The principled choice of
    # scale is notebook 4 — here we deliberately try a too-small (0.1) and a too-large (1.0) one.
    rng = np.random.default_rng(seed)
    return [
        {"W": rng.standard_normal((a, b)) * scale, "b": np.zeros(b)}
        for a, b in zip(sizes[:-1], sizes[1:], strict=True)
    ]


def activate(z, name):
    if name == "sigmoid":
        return 1.0 / (1.0 + np.exp(-z))
    if name == "tanh":
        return np.tanh(z)
    return np.maximum(0.0, z)  # relu


def deriv_from_output(a, name):
    # The activation's slope, written in terms of its OUTPUT a (cached during the forward pass).
    if name == "sigmoid":
        return a * (1.0 - a)
    if name == "tanh":
        return 1.0 - a ** 2
    return (a > 0).astype(float)  # relu


def softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def forward(params, X, hidden):
    acts, a, last = [X], X, len(params) - 1
    for i, layer in enumerate(params):
        z = a @ layer["W"] + layer["b"]
        a = softmax(z) if i == last else activate(z, hidden)
        acts.append(a)
    return acts


def backward(params, acts, y, hidden):
    # Exactly notebook 1/2's chain-rule loop; we keep the gradient w.r.t. EACH layer's weights.
    n = len(y)
    y_onehot = np.zeros_like(acts[-1])
    y_onehot[np.arange(n), y] = 1.0
    d = (acts[-1] - y_onehot) / n
    grads = [None] * len(params)
    for i in reversed(range(len(params))):
        grads[i] = acts[i].T @ d
        if i > 0:
            d = (d @ params[i]["W"].T) * deriv_from_output(acts[i], hidden)
    return grads


# A deep stack: 10 hidden layers of width 16 (so 11 weight matrices), on standardized circles.
X, y = make_circles(n_samples=400, noise=0.1, factor=0.4, random_state=SEED)
X = StandardScaler().fit_transform(X)
sizes = [2] + [16] * 10 + [2]


def per_layer_rms(hidden, scale):
    params = init_params(sizes, scale)
    acts = forward(params, X, hidden)
    grad_rms = [np.sqrt(np.mean(g ** 2)) for g in backward(params, acts, y, hidden)]
    fwd_rms = [np.sqrt(np.mean(acts[i] ** 2)) for i in range(1, len(params))]  # hidden layers
    return fwd_rms, grad_rms


fwd_vanish, grad_vanish = per_layer_rms("sigmoid", scale=0.1)
fwd_explode, grad_explode = per_layer_rms("relu", scale=1.0)

print("sigmoid + small weights (vanish):")
print(f"  gradient RMS: input layer {grad_vanish[0]:.2e}  ->  output layer {grad_vanish[-1]:.2e}")
print(f"  forward activation RMS: flat near {np.mean(fwd_vanish):.2f}")
print("ReLU + large weights (explode):")
print(f"  gradient RMS: ~{grad_explode[0]:.0e} at every layer")
print(f"  forward activation RMS: {fwd_explode[0]:.1f}  ->  {fwd_explode[-1]:.0f}")

In [ ]:
layers_fwd = np.arange(1, 11)        # 10 hidden layers
layers_grad = np.arange(1, 12)       # 11 weight matrices

fig, (axf, axg) = plt.subplots(1, 2, figsize=(14, 5))

axf.plot(layers_fwd, fwd_vanish, "o-", color=colors.COLORS["model"], lw=2,
         label="sigmoid + small")
axf.plot(layers_fwd, fwd_explode, "s-", color=colors.COLORS["error"], lw=2,
         label="ReLU + large")
axf.set_yscale("log")
axf.set_xlabel("hidden layer (1 = first)")
axf.set_ylabel("forward activation RMS (log scale)")
axf.set_title("Forward: the signal collapses or explodes")
axf.legend()

axg.plot(layers_grad, grad_vanish, "o-", color=colors.COLORS["model"], lw=2,
         label="sigmoid + small")
axg.plot(layers_grad, grad_explode, "s-", color=colors.COLORS["error"], lw=2,
         label="ReLU + large")
axg.set_yscale("log")
axg.set_xlabel("layer (1 = closest to input)")
axg.set_ylabel("gradient RMS (log scale)")
axg.set_title("Backward: the gradient vanishes or explodes")
axg.legend()

fig.tight_layout()
plt.show()

**Read the figure.** Two failure modes, side by side, on log axes.

*Vanish* (blue, sigmoid + small weights): the **gradient** falls about twelve orders of magnitude
from the output layer down to roughly `1e-13` at the input layer — the early layers receive
essentially zero learning signal. On the left, the **forward** activations flatline near `0.5`: every
sigmoid unit is pinned mid-range, so the layers carry almost no information forward either.

*Explode* (coral, ReLU + large weights): the **forward** activations climb from about `0.8` to nearly
`8000` across ten layers, and the **gradient** sits around `1000` everywhere — astronomically larger
than a healthy value (which would be near `1e-2`). Either way, the early layers are unusable: frozen
in one case, blown up in the other.

## Why it happens

**Sigmoid vanishes** because its slope is never large: `σ'(z) = σ(z)(1 − σ(z)) ≤ 0.25` for *every*
input (chapter 11, NB 4 — the saturating tails). Each layer therefore multiplies the backward signal
by at most about a quarter. Ten layers means a factor of at most `0.25¹⁰ ≈ 1e-6`, and the small
weights push it far lower still — down to the `1e-13` we measured.

**Large weights explode** because, with ReLU, the slope is 0 or 1 (no shrinking), but standard-normal
weights *scale the signal up* at every layer. Each layer multiplies by a factor well above 1, so the
product grows without bound — both forward (the activations) and backward (the gradient).

The lever, in both cases, is the **size of the per-layer factor**. There is a weight scale that keeps
each factor close to 1, so the signal neither dies nor explodes across depth — finding it is exactly
what **notebook 4 (initialization)** is about. First, let's confirm the damage in a real training run.

## The consequence: a network that cannot learn

If the early layers receive a gradient of `1e-13`, they never move from their random starting weights
— the network is effectively frozen. To see it end to end, we hand the job to scikit-learn's own
`MLPClassifier`: a network **five hidden layers deep**, trained on the same circles, with three
different activations. We read each run's **loss curve** (how the training loss falls epoch by epoch)
and its final accuracy.

In [ ]:
deep = (16, 16, 16, 16, 16)  # five hidden layers
curves = {}
for activation in ("logistic", "tanh", "relu"):
    clf = MLPClassifier(hidden_layer_sizes=deep, activation=activation,
                        max_iter=2000, random_state=SEED)
    clf.fit(X, y)
    curves[activation] = clf.loss_curve_
    print(f"{activation:9}: loss {clf.loss_curve_[0]:.4f} -> {clf.loss_curve_[-1]:.4f}   "
          f"accuracy {clf.score(X, y):.3f}   (ln 2 = {np.log(2):.4f})")

In [ ]:
names = {"logistic": "sigmoid", "tanh": "tanh", "relu": "ReLU"}
cmap = {"logistic": colors.COLORS["model"], "tanh": colors.COLORS["class_c"],
        "relu": colors.COLORS["error"]}

fig, ax = plt.subplots(figsize=(8.5, 5))
for activation, curve in curves.items():
    ax.plot(curve, color=cmap[activation], lw=2, label=f"{names[activation]} (5 layers)")
ax.axhline(np.log(2), color=colors.COLORS["muted"], ls="--", lw=1,
           label="ln 2 (knowing nothing)")
ax.set_xlabel("epoch")
ax.set_ylabel("training loss")
ax.set_title("Five layers deep: the sigmoid network never starts learning")
ax.legend()
plt.show()

**Read the figure.** The sigmoid network's loss (blue) sits flat at `ln 2 ≈ 0.69` — chance for these
two-class circles (the `ln K` "knowing nothing" baseline of notebook 1, here with K = 2) — and never
moves: its early layers get no gradient, so it stays at its random initialization and predicts chance
(accuracy 0.500). The tanh and ReLU networks
(green, coral) descend to near zero and reach perfect accuracy. Same depth, same data, same optimizer
— the only difference is how badly the activation saturates, and that alone decides whether the
network can learn at all.

## Is it only a bad optimizer?

A fair challenge: maybe the sigmoid network stalled only because of the particular optimizer (`adam`).
In chapter 11 a *shallow* sigmoid network was perfectly fine — a single hidden layer trained with the
`lbfgs` solver reached 100% on these circles. So let's give the deep sigmoid network that same solver,
the one that rescued it when it was shallow, and compare one hidden layer against five.

In [ ]:
for n_layers in (1, 5):
    clf = MLPClassifier(hidden_layer_sizes=(16,) * n_layers, activation="logistic",
                        solver="lbfgs", max_iter=2000, random_state=SEED)
    clf.fit(X, y)
    print(f"sigmoid, {n_layers} hidden layer(s), lbfgs: accuracy {clf.score(X, y):.3f}")

for activation in ("tanh", "relu"):
    clf = MLPClassifier(hidden_layer_sizes=(16,) * 5, activation=activation,
                        solver="lbfgs", max_iter=2000, random_state=SEED)
    clf.fit(X, y)
    print(f"{activation:4}, 5 hidden layers, lbfgs: accuracy {clf.score(X, y):.3f}")

**Read the result.** With `lbfgs`, one sigmoid layer still reaches 1.0 — but five sigmoid layers
collapse to chance (≈ 0.5). The solver that rescued the shallow network cannot rescue the deep one:
**depth, not the optimizer, is the cause.** Once the gradient has vanished, no choice of optimizer can
recover a signal that is no longer there. And notice the activation still matters — tanh and ReLU
clear five layers easily here, because they saturate far less than sigmoid. (Stack enough layers and
even they would struggle; the general cure is what comes next.)

## What you built

- You traced the gradient to an early layer and saw it is a **product of one factor per layer** — so a
  deep network either **vanishes** or **explodes** its own gradient.
- You **measured** both, by hand, through a ten-layer stack: a sigmoid net whose gradient fell to
  `1e-13`, and a large-weight ReLU net whose activations blew past `1000`.
- You watched the consequence in a real five-layer network: a sigmoid loss curve pinned at `ln 2`,
  learning nothing — and confirmed that **depth**, not the optimizer, is to blame.

This is the pivot of the chapter. Everything that follows is a way to keep the per-layer factor close
to 1, so the signal survives all the way down.

## Where this goes next

The diagnosis points straight at the cure. If the trouble is that each layer's factor drifts away from
1, then choosing the **initial weight scale** so those factors start near 1 should keep the gradient
alive across depth. That is **initialization** — notebook 4 (He and Xavier) — the direct fix for the
problem we diagnosed here. (Normalization, notebook 6, is a second lever on the same problem.)

## Your turn

1. **(warm-up)** Deepen the sigmoid stack from 10 to 20 hidden layers and re-measure the input-layer
   gradient RMS. How much smaller does it get? (Expect it to fall even further — toward `1e-24`.)
2. **(core)** Switch the exploding setup to **ReLU with small weights** (scale 0.1) instead of large.
   Does it still explode? What happens to the gradient instead? (This is your first hint that the
   *scale* is the whole lever — the subject of notebook 4.)
3. **(reach)** Read `loss_curve_` for the 5-layer sigmoid network and print its first few values.
   Confirm the loss sits at `ln 2` from the very first epoch — it never even starts.

## References

- Hochreiter, S. (1991). *Untersuchungen zu dynamischen neuronalen Netzen* (diploma thesis, TU
  München) — the original diagnosis of the vanishing gradient.
- Bengio, Y., Simard, P., & Frasconi, P. (1994). Learning long-term dependencies with gradient descent
  is difficult. *IEEE Transactions on Neural Networks* 5(2):157–166.
  https://doi.org/10.1109/72.279181
- Glorot, X., & Bengio, Y. (2010). Understanding the difficulty of training deep feedforward neural
  networks. *PMLR* 9:249–256 — the analysis behind the initialization fix (notebook 4).
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*, chapter 8. MIT Press.

*Previous:* **12.2 — depth is a representation hierarchy.**  *Next:* **12.4 — initialization: He &
Xavier.**